### HalfCheetah-Vel, HOW to use it
Upload your codes as a dataset in a zip file, naming dataset as 'codes'

In [ ]:
!cp -r /kaggle/input/datasets/<your kaggle username>/codes/* /kaggle/working/

In [ ]:
!pip install stable-baselines3
!pip install tyro loguru
!pip install mujoco

### check all the libraruies exist afters installation

In [ ]:
import torch
import gymnasium
import stable_baselines3
import mujoco
import tyro
import loguru
import tabulate
import wandb

print("torch:", torch.__version__)
print("gymnasium:", gymnasium.__version__)
print("sb3:", stable_baselines3.__version__)
print("mujoco:", mujoco.__version__)
print("tyro:", tyro.__version__)
print("loguru:", loguru.__version__)
print("tabulate:", tabulate.__version__)
print("wandb:", wandb.__version__)

In [ ]:
import os

os.chdir('/kaggle/working')

import run_continual_benchmark as benchmark

# ============================================================
# اورراید سریع برای یک تستِ کوچیک - بدون نیاز به آپلود دوباره‌ی کد
# ============================================================
benchmark.TASK_SEQUENCE = [2, 2]          # همون یک سرعت، دوبار (برای trigger شدن merge)
benchmark.POOL_SIZE = 1                    # مطمئن میشه merge حتماً یک‌بار اتفاق بیفته
benchmark.TOTAL_TIMESTEPS_PER_TASK = 400_000   # به‌اندازه‌ی کافی بزرگ که واقعاً ببینید converge میشه یا نه
benchmark.DISTILL_EXTRA_STEPS = 15_000
benchmark.LEARNING_STARTS = 5_000

if __name__ == "__main__":
    pass  # جلوگیری از اجرای دوباره‌ی __main__ فایل موقع import

device = benchmark.torch.device("cuda" if benchmark.torch.cuda.is_available() else "cpu")

all_prev_units = {}
for cond_name, cfg in benchmark.CONDITIONS.items():
    print(f"\n========== {cond_name} ==========")
    all_prev_units[cond_name] = benchmark.run_task_chain(cond_name, cfg)

for seq_idx, task_id in enumerate(benchmark.TASK_SEQUENCE):
    benchmark.plot_during_training(seq_idx, task_id, benchmark.get_task_name(task_id))

benchmark.plot_retention(all_prev_units, device)
print(f"\n*** Done. Plots in {benchmark.PLOTS_ROOT} ***")